# Minibatch SGD as a first-class primitive

`pycograd_batching_example.ipynb` trained on minibatches, but the batching was *hand-rolled inside the objective*:

```python
obj = |> rng.choice(N, size=40, replace=False) |> push |> X[$] |> logits |> pop *|> cross_entropy($, Y[$])
for _ in range(steps):
    value, grads = weights.grad(obj)
    weights.step(grads, 0.5)
```

Three concerns were tangled together: **sampling** (`rng.choice`), **threading the sampled indices** so inputs and labels stay aligned (the `push` / `pop` global stack, rejoined with `*|>`), and a **manual loop** with a hardcoded learning rate.

This demo uses the first-class primitives instead:

* `weights.grad(objective, batch)` now forwards `batch` to the objective, so the **objective is a pure function of a minibatch** — no sampling or index threading inside it.
* `fit(weights, objective, X, Y, ...)` drives the existing `batches` / `DataLoader` minibatcher and an optimizer over the ambient weights.
* `weights.step(grads, opt)` accepts a plain learning rate **or** any `Optimizer` (`SGD`, `Adam`, ...).

Sampling goes back to `batches` (where it belongs), `push`/`pop` disappear (one shared index keeps `X` and `Y` aligned for free), and the loop becomes `epochs` + an optimizer.

In [1]:
%load_ext pycograd

In [2]:
import numpy as np
from pycograd import cross_entropy, relu, softmax
from pycograd import fit, accuracy, SGD, Adam, cosine_decay

rng = np.random.default_rng(42)

# Three Gaussian blobs, one per class.
m = 60
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.6, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]  # one-hot, 3 classes

# Hold out a fifth of the data for validation.
perm = rng.permutation(len(X))
n_val = len(X) // 5
val_idx, tr_idx = perm[:n_val], perm[n_val:]
Xtr, Ytr, ytr = X[tr_idx], Y[tr_idx], labels[tr_idx]
Xval, yval = X[val_idx], labels[val_idx]
len(Xtr), len(Xval)

(144, 36)

## A parameterized objective + `fit`

`logits` is the usual `$`-seeded forward pipe. The **loss** is now a function of one minibatch `(Xb, Yb)`: the tuple is splatted with `*|>` into named holes `($x, $y)`, then into the loss expression. `fit` slices `Xtr` / `Ytr` into aligned minibatches each epoch, feeds each through `weights.grad`, and steps `SGD`. It returns the per-epoch mean loss and calls `on_epoch` for logging.

In [3]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    logits  = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    forward = logits .> softmax
    loss    = $ *|> ($x, $y) *|> cross_entropy(logits($x), $y)
    history = fit(
        weights, loss, Xtr, Ytr,
        epochs=60, batch_size=32, opt=SGD(lr=0.5),
        shuffle=True, rng=rng,
        on_epoch=lambda e, l: e % 15 == 0 and print(f"epoch {e:2d}   loss {l:.4f}"),
    )

epoch  0   loss 0.3589
epoch 15   loss 0.0030


epoch 30   loss 0.0014
epoch 45   loss 0.0009


In [4]:
print(f"epochs run:    {len(history)}")
print(f"loss:          {history[0]:.4f} -> {history[-1]:.4f}")
print(f"train accuracy: {accuracy(forward(Xtr), ytr):.3f}")
print(f"  val accuracy: {accuracy(forward(Xval), yval):.3f}")

epochs run:    60
loss:          0.3589 -> 0.0007
train accuracy: 1.000
  val accuracy: 1.000


## Swapping the optimizer: Adam + a cosine learning-rate schedule

Because `fit` applies the update through `weights.step(grads, opt)`, any `Optimizer` drops in unchanged — here `Adam` with a `cosine_decay` schedule (`lr` may be a float or a `callable(step) -> lr`). Nothing about the model or the objective changes; only `opt` does. (`fit` deliberately has no `jit` knob: jit caches the trace and would freeze the first minibatch — use the full-batch `train` for jit.)

In [5]:
epochs = 60
steps_per_epoch = -(-len(Xtr) // 32)  # ceil division
schedule = cosine_decay(0.05, total_steps=epochs * steps_per_epoch)

with params{
    h1 = 0.3 * rng.standard_normal((2, 16))
    d1 = np.zeros(16)
    h2 = 0.3 * rng.standard_normal((16, 3))
    d2 = np.zeros(3)
} as model:
    logits2  = $ |> $ @ h1 + d1 |> relu |> $ @ h2 + d2
    forward2 = logits2 .> softmax
    loss2    = $ *|> ($x, $y) *|> cross_entropy(logits2($x), $y)
    history2 = fit(
        model, loss2, Xtr, Ytr,
        epochs=epochs, batch_size=32, opt=Adam(lr=schedule),
        shuffle=True, rng=rng,
    )

print(f"Adam loss:      {history2[0]:.4f} -> {history2[-1]:.4f}")
print(f"train accuracy: {accuracy(forward2(Xtr), ytr):.3f}")
print(f"  val accuracy: {accuracy(forward2(Xval), yval):.3f}")

Adam loss:      0.6422 -> 0.0000
train accuracy: 1.000
  val accuracy: 1.000


## Notes

* **One minibatch in, one scalar out.** `fit` hands the objective each minibatch as a single argument — a `(Xb, Yb)` tuple here — which the pipe splats with `*|>`. A single-array dataset (`fit(weights, obj, X, ...)`) hands a bare array, so the objective is just a `$`-seeded pipe.
* **`batches` does the sampling.** `fit` builds a fresh shuffled `DataLoader` internally; pass a prebuilt `DataLoader` instead (`fit(weights, obj, loader, epochs=..., opt=...)`) when you want `drop_last` or a custom rng. Inputs and labels are sliced with one shared index, so they stay aligned with no `push`/`pop`.
* **Optimizer state persists.** Reusing the same `Optimizer` across `fit` calls resumes its momentum / Adam moments and step count; pass a fresh optimizer to restart.
* **`backend=`** is forwarded to `weights.grad`, so the very same minibatch loop trains on torch / jax / tf.